# Mission 3 — 2-8: 기존 최고 앙상블 + 두 창 문맥 모델

목표: 내부 검증 Macro F1 0.66, 임계값 0.5 고정.
2-5의 최고 앙상블(0.650944)에 2-7의 앞·뒤 문맥 모델(0.637320)을 추가합니다.
단독 성능이 낮아도 다른 오류를 보완할 수 있다는 가설입니다. 향상을 보장하지 않습니다.

**단 하나의 신규 조건: 2-5 예측 확률 0.8 + 2-7 예측 확률 0.2.**
임계값이나 결합 비율 탐색은 없습니다. 기존 두 모델의 단독 점수는 비교용입니다.
2-5 내의 결합 비율은 그대로 재사용합니다. 전체 조합은 기존 네 RoBERTa 모델과 TF-IDF,
새 문맥 RoBERTa 모델을 필요로 합니다. 실사용 비용도 함께 고려해야 합니다.

업로드 파일 2개:
1. `m3_2_5_tfidf_weight_results.tar`
2. `m3_2_7_base_seed42_results.tar`

CPU 런타임으로 실행할 수 있습니다. 학습 ZIP이나 GPU는 필요하지 않습니다.
같은 내부 검증셋을 반복 사용한 탐색 결과이며 독립 최종 평가가 아닙니다.
전체 셀 실행 후 `m3_2_8_context_blend_results.tar`를 다운로드하세요.


## 1. 사전 고정 설정 및 입력 확인


In [ ]:
from pathlib import Path
import io,json,tarfile,csv,tempfile,hashlib,platform
from datetime import datetime,timezone
import numpy as np
ROOT=Path('/content')
BEST=ROOT/'m3_2_5_tfidf_weight_results.tar'
CONTEXT=ROOT/'m3_2_7_base_seed42_results.tar'
for p in [BEST,CONTEXT]:
    assert p.is_file(),f'Colab 파일 패널에서 업로드하세요: {p}'
TARGETS=['고열','구토','두통','복통','어지러움','열상','오심','전신쇠약','호흡곤란']
BEST_WEIGHT=.8
CONTEXT_WEIGHT=.2
THRESHOLD=.5
OUT=Path(tempfile.mkdtemp(prefix='m3_2_8_',dir=ROOT))
plan={'best_weight':BEST_WEIGHT,'context_weight':CONTEXT_WEIGHT,'threshold':THRESHOLD,
      'target_macro_f1':.66,'new_conditions':1,'evaluation':'internal validation',
      'created_utc':datetime.now(timezone.utc).isoformat()}
(OUT/'experiment_plan.json').write_text(json.dumps(plan,indent=2))
print('결과 폴더:',OUT)


## 2. 예측 배열과 정답 일치 검사


In [ ]:
def load(archive_path,member):
    with tarfile.open(archive_path,'r:*') as archive:
        item=archive.getmember(member)
        assert item.isfile() and item.size<20_000_000
        raw=archive.extractfile(item).read()
    with np.load(io.BytesIO(raw),allow_pickle=False) as z:
        result={k:z[k].copy() for k in ['probs','y_true','file_names','targets']}
    assert result['probs'].shape==result['y_true'].shape==(5792,9)
    assert np.isfinite(result['probs']).all()
    assert ((result['probs']>=0)&(result['probs']<=1)).all()
    assert np.isin(result['y_true'],[0,1]).all()
    assert list(result['targets'])==TARGETS
    assert len(np.unique(result['file_names']))==5792
    return result,hashlib.sha256(raw).hexdigest()
best,best_hash=load(BEST,'B80_T1_20_predictions.npz')
context,context_hash=load(CONTEXT,'seed_42/w1_encoder/dev_predictions.npz')
for key in ['y_true','targets','file_names']:
    assert np.array_equal(best[key],context[key]),f'자료 불일치: {key}'
with tarfile.open(BEST,'r:*') as archive:
    previous=json.load(archive.extractfile('source_provenance.json'))
with tarfile.open(CONTEXT,'r:*') as archive:
    config=json.load(archive.extractfile('seed_42/w1_encoder/run_config.json'))
assert previous['manifest_sha256']==config['manifest_sha256']
assert config['selection']=='final_epoch' and config['threshold']==.5
assert config['input']=='first_last_512_logit_mean'
(OUT/'source_provenance.json').write_text(json.dumps({
    'best_archive':BEST.name,'best_prediction_sha256':best_hash,
    'context_archive':CONTEXT.name,'context_prediction_sha256':context_hash,
    'manifest_sha256':config['manifest_sha256'],'context_run_config':config,
    'best_source_provenance':previous},ensure_ascii=False,indent=2))
print('검증 5,792건의 정답·행 순서·타깃·분할 일치 확인')


## 3. 고정 임계값 평가와 예측 결합


In [ ]:
def metrics(y,probs):
    truth=np.asarray(y,dtype=bool);pred=np.asarray(probs)>=THRESHOLD
    assert truth.shape==pred.shape
    tp=(truth&pred).sum(0);fp=(~truth&pred).sum(0);fn=(truth&~pred).sum(0)
    def div(a,b):return np.divide(a,b,out=np.zeros_like(a,dtype=float),where=b!=0)
    f=div(2*tp,2*tp+fp+fn)
    return {'macro_f1':float(f.mean()),'precision':div(tp,tp+fp),'recall':div(tp,tp+fn),
            'f1':f,'tp':tp,'fp':fp,'fn':fn,'support':truth.sum(0)}
assert metrics([[1,0],[0,1]],[[.5,.1],[.1,.9]])['macro_f1']==1.
assert metrics([[0,0]],[[0.,0.]])['macro_f1']==0.
y=best['y_true']
probs=BEST_WEIGHT*best['probs'].astype(np.float64)+CONTEXT_WEIGHT*context['probs'].astype(np.float64)
predictions={'best_2_5':best['probs'],'context_2_7':context['probs'],'best80_context20':probs}
evaluated={name:metrics(y,p) for name,p in predictions.items()}
assert abs(evaluated['best_2_5']['macro_f1']-0.650944375897425)<1e-9
assert abs(evaluated['context_2_7']['macro_f1']-0.6373197067622774)<1e-9
summary=[];labels=[]
for name,result in evaluated.items():
    summary.append({'experiment':name,'macro_f1':result['macro_f1'],
        'delta_vs_2_5':result['macro_f1']-evaluated['best_2_5']['macro_f1'],
        'threshold':THRESHOLD,'target_reached':result['macro_f1']>=.66})
    for i,symptom in enumerate(TARGETS):
        labels.append({'experiment':name,'symptom':symptom,
            **{k:float(result[k][i]) for k in ['precision','recall','f1']},
            **{k:int(result[k][i]) for k in ['tp','fp','fn','support']}})
def save_csv(path,rows):
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
save_csv(OUT/'summary.csv',summary);save_csv(OUT/'per_label.csv',labels)
np.savez_compressed(OUT/'best80_context20_predictions.npz',probs=probs,y_true=y,
    file_names=best['file_names'],targets=best['targets'])
for row in summary:
    print(f"{row['experiment']:22s} F1={row['macro_f1']:.6f} 차이={row['delta_vs_2_5']:+.6f}")
print('목표 0.66 달성:',evaluated['best80_context20']['macro_f1']>=.66)


## 4. 보고서와 결과 TAR 다운로드


In [ ]:
lines=['# 2-8 문맥 모델 추가 결합 결과','',
    '내부 검증 5,792건, 임계값 0.5, 2-5 80% + 2-7 20% 사전 고정.', '',
    '|실험|Macro F1|2-5 대비|','|---|---:|---:|']
for r in summary:lines.append(f"|{r['experiment']}|{r['macro_f1']:.6f}|{r['delta_vs_2_5']:+.6f}|")
lines+=['','같은 내부 검증셋에서 반복한 탐색 결과이며 독립 평가가 아니다.',
        '새 데이터 추론에는 구성 모델의 가중치와 동일 전처리가 필요하다.',
        '이 TAR에는 모델 가중치가 없다. 예측 npz에는 통화 파일명이 포함되므로 Git에는 집계 지표만 올린다.']
(OUT/'REPORT.md').write_text('\n'.join(lines),encoding='utf-8')
(OUT/'environment.json').write_text(json.dumps({'python':platform.python_version(),'numpy':np.__version__},indent=2))
destination=ROOT/'m3_2_8_context_blend_results.tar'
with tarfile.open(destination,'w') as archive:
    for p in sorted(OUT.iterdir()):
        if p.is_file():archive.add(p,arcname=p.name)
print('결과:',destination)
from google.colab import files
files.download(str(destination))
